# Qwen-Image-Edit 2511 + NSFW LoRA (MCNL v1) — Colab runner for a free T4
Runs the **plain Qwen-Image-Edit-2511** base model (GGUF, small enough for a T4) and adds
`ScottzillaSystems/qwen-image-edit-plus-nsfw-lora` on top. Text-to-image **and** image editing.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (L4 / A100 are faster).

Run the 3 cells in order (cells 1 and 2 only once per session):
1. **Setup** – installs ComfyUI, downloads the base model, text encoder, VAE, the NSFW LoRA and the speed (Lightning) LoRAs (~22 GB)
2. **Start engine** – launches ComfyUI and checks that every file/node is found
3. **Generate** – leave `input_image_path` empty for text-to-image, or paste an image path to edit it

To get an image path: upload the image in the left sidebar (📁 Files), right-click it → **Copy path**.

**Important:** the base model is *not* uncensored by itself — the NSFW LoRA is what enables it, and it reacts to
trigger words in the prompt: `nsfw` `nipples` `vagina` `penis` `missionary` `cowgirlout` `reversecowgirlpov` `blowjob` `cum_on_face` `creamp1e` `l1ck`.

**Speed:** without a speed LoRA this model needs ~40 steps and takes very long on a T4. The default uses the
Lightning 4-step LoRA (about 10× faster).

In [ ]:
#@title 1️⃣ Setup: install ComfyUI + download all files (run once, ~15-20 min)
base_quant = "Q4_K_M" #@param ["Q3_K_S","Q3_K_M","Q4_0","Q4_K_S","Q4_K_M","Q5_K_M","Q6_K","Q8_0"]
#@markdown Free T4 (15 GB): use Q4_0 / Q4_K_S (safest) or Q4_K_M (best quality that still fits, ~13 GB). Q5 and above need an L4 / A100.

import os, re, sys, json, shutil, subprocess, time

def sh(cmd, cwd=None, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError("Command failed: " + cmd)
    return r.returncode

PIP = sys.executable + " -m pip install -q "

# ---------- GPU / RAM ----------
g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                   shell=True, capture_output=True, text=True)
if g.returncode != 0 or not g.stdout.strip():
    raise RuntimeError("No GPU found. Runtime > Change runtime type > pick a GPU, then run this cell again.")
m = re.search(r"(\d+)\s*MiB", g.stdout)
gpu_mb = int(m.group(1)) if m else 0
ram_gb = int(open("/proc/meminfo").readline().split()[1]) / 1024 / 1024
print("GPU:", g.stdout.strip(), "| system RAM: %.1f GB" % ram_gb)

# ---------- ComfyUI ----------
if not os.path.exists("/content/ComfyUI/main.py"):
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI")
else:
    sh("git pull", cwd="/content/ComfyUI", check=False)
sh(PIP + "-r requirements.txt", cwd="/content/ComfyUI")

# ---------- GGUF loader (needed for the GGUF text encoder) ----------
for old in ["ComfyUI-GGUF", "comfyui-gguf"]:
    shutil.rmtree("/content/ComfyUI/custom_nodes/" + old, ignore_errors=True)
cn = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF-Loader"
if not os.path.exists(cn):
    sh("git clone --depth 1 https://github.com/ChrisColeTech/ComfyUI-GGUF-Loader " + cn)
else:
    sh("git pull", cwd=cn, check=False)
if os.path.exists(cn + "/requirements.txt"):
    sh(PIP + "-r requirements.txt", cwd=cn)
sh(PIP + "--upgrade gguf")

# ---------- make sure the Python packages agree with each other ----------
def deps_ok():
    r = subprocess.run([sys.executable, "-c", "import transformers, huggingface_hub, gguf, PIL, requests"],
                       capture_output=True, text=True)
    return r.returncode == 0, r.stderr[-600:]

ok, err = deps_ok()
if not ok:
    print("Dependency conflict detected, repairing...\n", err)
    sh(PIP + '"huggingface_hub>=1.5.0,<2.0"', check=False)
    ok, err = deps_ok()
    if not ok:
        sh(PIP + "-U transformers", check=False)
        ok, err = deps_ok()
if not ok:
    raise RuntimeError("Package conflict could not be repaired:\n" + err)
print("Python packages OK ✅")

# ---------- find the exact files in each repo ----------
from huggingface_hub import HfApi, hf_hub_download

def list_repo(repo):
    try:
        info = HfApi().model_info(repo, files_metadata=True)
        return {s.rfilename: (s.size or 0) for s in info.siblings}
    except Exception as e:
        raise RuntimeError("Could not read repo listing for %s: %s" % (repo, repr(e)[:300]))

BASE_REPO  = "unsloth/Qwen-Image-Edit-2511-GGUF"
ENC_REPO   = "ChrisColeTech/qwen-image-edit-uncensored-GGUF"
LIGHT_REPO = "lightx2v/Qwen-Image-Edit-2511-Lightning"
NSFW_REPO  = "ScottzillaSystems/qwen-image-edit-plus-nsfw-lora"

base_files, enc_files, light_files, nsfw_files = [list_repo(r) for r in (BASE_REPO, ENC_REPO, LIGHT_REPO, NSFW_REPO)]

# base model: root-level .gguf whose name contains the chosen quant (shortest name wins)
cands = [f for f in base_files if f.endswith(".gguf") and "/" not in f and "mmproj" not in f.lower()
         and base_quant.lower() in f.lower()]
if not cands:
    avail = sorted(f for f in base_files if f.endswith(".gguf") and "/" not in f)
    raise RuntimeError("No file for %s in %s. Available: %s" % (base_quant, BASE_REPO, avail))
base_file = sorted(cands, key=len)[0]

# text encoder + mmproj + vae (from the ChrisColeTech v1 repo)
te_file   = "split/text_encoders/Qwen2.5-VL-7B-Instruct-q4_0.gguf"
mmproj    = "split/text_encoders/Qwen2.5-VL-7B-Instruct-mmproj-f16.gguf"
vae_file  = "split/vae/qwen_image_vae.safetensors"
for f in (te_file, mmproj, vae_file):
    if f not in enc_files:
        raise RuntimeError("Expected file missing in %s: %s" % (ENC_REPO, f))

# Lightning speed LoRAs
def find_lightning(steps):
    hits = [f for f in light_files if ("lightning-%dsteps" % steps) in f.lower() and f.lower().endswith(".safetensors")
            and "fp8" not in f.lower() and "/" not in f]
    bf = [f for f in hits if "bf16" in f.lower()]
    return (bf or hits or [None])[0]
light4, light8 = find_lightning(4), find_lightning(8)
if not light4:
    raise RuntimeError("Could not find the Lightning 4-step LoRA in " + LIGHT_REPO)

# NSFW LoRA
nsfw_lora = "qwen-image-edit-plus-nsfw-lora.safetensors"
if nsfw_lora not in nsfw_files:
    others = [f for f in nsfw_files if f.endswith(".safetensors")]
    if not others:
        raise RuntimeError("No .safetensors LoRA found in " + NSFW_REPO)
    nsfw_lora = others[0]

plan = [
    (BASE_REPO, base_file, base_files, "diffusion_models"),
    (ENC_REPO, te_file, enc_files, "text_encoders"),
    (ENC_REPO, mmproj, enc_files, "text_encoders"),
    (ENC_REPO, vae_file, enc_files, "vae"),
    (LIGHT_REPO, light4, light_files, "loras"),
    (NSFW_REPO, nsfw_lora, nsfw_files, "loras"),
]
if light8:
    plan.append((LIGHT_REPO, light8, light_files, "loras"))

need_gb = sum(files[f] for _, f, files, _ in plan) / 1e9
free_gb = shutil.disk_usage("/content").free / 1e9
print("\nDownload plan (%.1f GB, free disk %.1f GB):" % (need_gb, free_gb))
for repo, f, files, sub in plan:
    print("  %-8s %-62s %5.2f GB" % (sub, os.path.basename(f), files[f] / 1e9))
if free_gb < need_gb + 3:
    raise RuntimeError("Not enough free disk space (%.1f GB free, %.1f GB needed)." % (free_gb, need_gb))
if gpu_mb < 20000 and base_quant.startswith(("Q5", "Q6", "Q8")):
    print("\n⚠️  %s is too big for a %d MB GPU — expect out-of-memory errors. Prefer Q4_K_S / Q4_0." % (base_quant, gpu_mb))

# ---------- download ----------
M = "/content/ComfyUI/models"

def fetch(repo, repo_file, files, sub):
    name = os.path.basename(repo_file)
    dest_dir = os.path.join(M, sub)
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 0.98 * files[repo_file]:
        print("✔ already downloaded:", name)
        return name
    os.makedirs(dest_dir, exist_ok=True)
    for attempt in range(1, 4):
        try:
            print("⬇ downloading", name, "(%.1f GB, attempt %d)..." % (files[repo_file] / 1e9, attempt))
            p = hf_hub_download(repo_id=repo, filename=repo_file, local_dir="/content/hf_dl")
            shutil.move(p, dest)
            print("  saved ->", dest)
            return name
        except Exception as e:
            print("  failed:", repr(e)[:300])
            time.sleep(5)
    raise RuntimeError("Could not download " + name + " after 3 tries. Run the cell again.")

names = {}
for repo, f, files, sub in plan:
    names[f] = fetch(repo, f, files, sub)

json.dump({
    "unet": names[base_file],
    "clip": names[te_file],
    "vae": names[vae_file],
    "nsfw_lora": names[nsfw_lora],
    "lightning4": names[light4],
    "lightning8": names[light8] if light8 else None,
}, open("/content/qwen_lora_cfg.json", "w"))
print("\n✅ Setup finished. Now run cell 2.")


In [ ]:
#@title 2️⃣ Start the ComfyUI engine (run after cell 1, ~1-2 min)
low_vram_mode = "auto" #@param ["auto","on","off"]
#@markdown "auto" = on for GPUs under 20 GB (free T4), off for L4 / A100.
restart_engine = False #@param {type:"boolean"}
#@markdown Tick `restart_engine` whenever you change the options in this cell.
extra_flags = "" #@param {type:"string"}
#@markdown Optional advanced flags, e.g. `--fp32-vae` (only if images come out black/garbled).

import os, sys, json, time, shlex, subprocess, requests

URL = "http://127.0.0.1:8188"

def alive():
    try:
        return requests.get(URL + "/system_stats", timeout=2).ok
    except Exception:
        return False

if restart_engine:
    subprocess.run("pkill -f 'main.py --listen'", shell=True)
    time.sleep(4)

if alive():
    print("Engine already running ✅")
else:
    os.makedirs("/content/outputs", exist_ok=True)
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--output-directory", "/content/outputs", "--disable-auto-launch"]
    gm = subprocess.run("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits",
                        shell=True, capture_output=True, text=True).stdout.strip().split("\n")[0]
    gpu_mb = int(gm) if gm.isdigit() else 0
    if low_vram_mode == "on" or (low_vram_mode == "auto" and gpu_mb < 20000):
        cmd.append("--lowvram")
        print("(low-VRAM mode on)")
    if extra_flags.strip():
        cmd += shlex.split(extra_flags)
    log = open("/content/comfyui.log", "w")
    subprocess.Popen(cmd, cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
    print("Starting engine", end="")
    started = False
    for _ in range(150):
        if alive():
            started = True
            break
        print(".", end="", flush=True)
        time.sleep(2)
    if not started:
        print("\n❌ Engine did not start. Last log lines:\n")
        print("".join(open("/content/comfyui.log").readlines()[-40:]))
        raise RuntimeError("ComfyUI failed to start (copy the log above and send it to get it fixed)")
    print("\nEngine started ✅")

# ---------- verify everything the generator needs ----------
cfg = json.load(open("/content/qwen_lora_cfg.json"))

def node_info(name):
    return requests.get(URL + "/object_info/" + name).json().get(name)

def options(name, field):
    info = node_info(name)
    if not info:
        return None
    spec = info["input"]["required"][field]
    if isinstance(spec[0], list):
        return spec[0]
    if len(spec) > 1 and isinstance(spec[1], dict) and "options" in spec[1]:
        return spec[1]["options"]
    return []

problems = []
for n in ["UnetLoaderGGUF", "CLIPLoaderGGUF", "LoraLoaderModelOnly", "ModelSamplingAuraFlow",
          "CLIPTextEncode", "KSampler", "EmptyLatentImage", "VAELoader", "VAEDecode", "SaveImage", "LoadImage"]:
    if node_info(n):
        print("✅ node", n)
    else:
        print("❌ node missing:", n)
        problems.append(n)

if node_info("TextEncodeQwenImageEditPlus") or node_info("TextEncodeQwenImageEdit"):
    print("✅ Qwen image-edit text-encode node")
else:
    print("❌ no TextEncodeQwenImageEdit(Plus) node — ComfyUI is too old")
    problems.append("TextEncodeQwenImageEdit")

rm = node_info("FluxKontextMultiReferenceLatentMethod")
if rm and "index_timestep_zero" in (options("FluxKontextMultiReferenceLatentMethod", "reference_latents_method") or []):
    print("✅ reference-latent method index_timestep_zero (recommended for 2511)")
else:
    print("⚠️ index_timestep_zero not available — editing still works but may drift more")

checks = [("UnetLoaderGGUF", "unet_name", cfg["unet"]),
          ("CLIPLoaderGGUF", "clip_name", cfg["clip"]),
          ("VAELoader", "vae_name", cfg["vae"]),
          ("LoraLoaderModelOnly", "lora_name", cfg["nsfw_lora"]),
          ("LoraLoaderModelOnly", "lora_name", cfg["lightning4"])]
for node, field, want in checks:
    opts = options(node, field)
    if opts is not None and want in opts:
        print("✅ file found:", want)
    else:
        print("❌ file NOT listed by", node, ":", want)
        problems.append(want)

types = options("CLIPLoaderGGUF", "type") or []
if "qwen_image" in types:
    print("✅ CLIP type qwen_image")
else:
    print("❌ CLIP type qwen_image not available")
    problems.append("qwen_image type")

if problems:
    print("\nLast log lines:\n" + "".join(open("/content/comfyui.log").readlines()[-40:]))
    raise RuntimeError("Something is missing: " + ", ".join(problems) +
                       ". Run cell 1 again, then tick restart_engine and run this cell again.")
print("\n✅ Everything is ready. Now run cell 3.")


In [ ]:
#@title 3️⃣ Generate / Edit image
prompt = "make him wear hat " #@param {type:"string"}
input_image_path = "/content/images.jpeg" #@param {type:"string"}
#@markdown **Empty = text-to-image.** Paste a path like `/content/photo.jpg` = edit that image with your prompt.
#@markdown ---
#@markdown **NSFW LoRA** (trigger words: nsfw, nipples, vagina, penis, missionary, cowgirlout, reversecowgirlpov, blowjob, cum_on_face, creamp1e, l1ck)
use_nsfw_lora = True #@param {type:"boolean"}
nsfw_lora_strength = 1.0 #@param {type:"slider", min:0.0, max:1.5, step:0.05}
add_nsfw_trigger = False #@param {type:"boolean"}
#@markdown `add_nsfw_trigger` puts "nsfw," in front of your prompt. Leave it off for ordinary edits (like sunglasses).
#@markdown ---
speed_lora = "Lightning 8-step (better quality)" #@param ["Lightning 4-step (fast, recommended)","Lightning 8-step (better quality)","None (40 steps - extremely slow on T4)"]
style = "Natural" #@param ["None","Natural","Cinematic","Photorealistic","3D Render","Animated / Cartoon","Anime","Digital Painting","Oil Painting","Watercolor","Pixel Art","Cyberpunk","Fantasy Art","Product Photo","Minimal Vector"]
#@markdown Style is added to the end of your prompt. Keep it `None` when editing an image.
aspect_ratio = "Match input image" #@param ["Match input image","1:1","16:9","9:16","4:3","3:4","3:2","2:3","21:9"]
megapixels = "1.0" #@param ["0.5","1.0","1.5","2.0"]
#@markdown 1.0 MP ≈ 1024×1024. If you get out-of-memory on a T4, use 0.5.
num_images = 1 #@param {type:"slider", min:1, max:8, step:1}
seed = -1 #@param {type:"integer"}
#@markdown seed = -1 means random.
#@markdown ---
#@markdown Advanced. `0` = automatic (4 / 8 / 40 steps and cfg 1 / 1 / 4 depending on the speed LoRA):
steps = 0 #@param {type:"integer"}
cfg = 0.0 #@param {type:"number"}
aura_shift = 3.0 #@param {type:"number"}
negative_prompt = " " #@param {type:"string"}
#@markdown Negative prompt is only used when cfg is above 1 (the "None" speed mode).

import os, io, json, math, random, time, requests
from PIL import Image, ImageOps
from IPython.display import display

URL = "http://127.0.0.1:8188"
OUT_DIR = "/content/outputs"

STYLES = {
    "None": "",
    "Natural": "natural photograph, soft natural daylight, true-to-life colors, realistic textures, candid look, no heavy filters",
    "Cinematic": "cinematic film still, dramatic lighting, shallow depth of field, anamorphic lens, subtle film grain, color graded",
    "Photorealistic": "ultra realistic photograph, natural lighting, sharp focus, highly detailed, 85mm lens",
    "3D Render": "high quality 3D render, octane render, soft global illumination, subsurface scattering, detailed materials",
    "Animated / Cartoon": "stylized 3D animated movie look, expressive, vibrant colors, clean shapes, soft lighting",
    "Anime": "anime style illustration, clean line art, vibrant colors, detailed background, cel shading",
    "Digital Painting": "digital painting, concept art, painterly brush strokes, rich colors, highly detailed",
    "Oil Painting": "classical oil painting, visible brush strokes, rich texture, museum quality",
    "Watercolor": "watercolor illustration, soft washes, paper texture, delicate colors",
    "Pixel Art": "pixel art, 16-bit retro game style, limited color palette, crisp pixels",
    "Cyberpunk": "cyberpunk aesthetic, neon lights, rainy night, futuristic city, high contrast",
    "Fantasy Art": "epic fantasy art, magical atmosphere, intricate detail, volumetric light",
    "Product Photo": "professional product photography, studio lighting, clean background, sharp focus",
    "Minimal Vector": "flat vector illustration, minimal design, bold shapes, clean colors",
}

def snap(v, m=16, lo=256, hi=2048):
    return int(min(hi, max(lo, round(v / m) * m)))

def size_from_ratio(ratio, mp):
    a, b = [int(x) for x in ratio.split(":")]
    area = float(mp) * 1024 * 1024
    h = math.sqrt(area * b / a)
    return snap(h * a / b), snap(h)

def size_from_image(iw, ih, mp):
    area = float(mp) * 1024 * 1024
    h = math.sqrt(area * ih / iw)
    return snap(h * iw / ih), snap(h)

def engine_alive():
    try:
        return requests.get(URL + "/system_stats", timeout=3).ok
    except Exception:
        return False

def log_tail(n=30):
    try:
        return "".join(open("/content/comfyui.log").readlines()[-n:])
    except Exception:
        return "(no log)"

def upload_image(path):
    im = Image.open(path)
    im = ImageOps.exif_transpose(im).convert("RGB")
    if max(im.size) > 2048:
        im.thumbnail((2048, 2048), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="PNG")
    buf.seek(0)
    name = "colab_input_%d.png" % int(time.time())
    r = requests.post(URL + "/upload/image",
                      files={"image": (name, buf, "image/png")},
                      data={"type": "input", "overwrite": "true"})
    if r.status_code != 200:
        raise RuntimeError("Could not hand the image to ComfyUI: " + r.text[:300])
    j = r.json()
    value = j["name"] if not j.get("subfolder") else j["subfolder"] + "/" + j["name"]
    return value, im

def wait_for(pid, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            h = requests.get(URL + "/history/" + pid, timeout=10).json()
        except Exception:
            print("❌ The engine stopped responding (usually it ran out of RAM). Log tail:\n" + log_tail())
            raise RuntimeError("Engine crashed. Run cell 2 with restart_engine ticked and low_vram_mode on, "
                               "then lower megapixels to 0.5 and try again.")
        if pid in h:
            st = h[pid].get("status", {})
            if st.get("status_str") == "error":
                for kind, data in st.get("messages", []):
                    if kind == "execution_error":
                        print("❌", data.get("exception_type"), "-", data.get("exception_message"))
                raise RuntimeError("Generation failed (message above). If it says out of memory: cell 2 -> "
                                   "low_vram_mode on + restart_engine, and use megapixels 0.5.")
            if h[pid].get("outputs"):
                return h[pid]
        time.sleep(2)
    raise TimeoutError("Timed out waiting for the image.")

def side_by_side(a, b, height=512):
    a2 = a.resize((max(1, int(a.width * height / a.height)), height))
    b2 = b.resize((max(1, int(b.width * height / b.height)), height))
    canvas = Image.new("RGB", (a2.width + b2.width + 10, height), (30, 30, 30))
    canvas.paste(a2, (0, 0))
    canvas.paste(b2, (a2.width + 10, 0))
    return canvas

# ---------------- checks ----------------
if not prompt.strip():
    raise ValueError("Please type a prompt.")
if not engine_alive():
    raise RuntimeError("The engine is not running. Run cell 2 first.")
if not os.path.exists("/content/qwen_lora_cfg.json"):
    raise RuntimeError("Setup was not run. Run cell 1 first.")
cfgfile = json.load(open("/content/qwen_lora_cfg.json"))

path = input_image_path.strip().strip('"').strip("'")
edit_mode = bool(path)
if edit_mode:
    path = os.path.expanduser(path)
    if not os.path.isfile(path):
        folder = os.path.dirname(path) or "/content"
        print("❌ File not found:", path)
        if os.path.isdir(folder):
            print("Files in", folder, ":", [f for f in os.listdir(folder)][:30])
        raise FileNotFoundError("Right-click your uploaded image in the 📁 Files sidebar -> Copy path, and paste it into input_image_path.")

# ---------------- speed LoRA -> steps / cfg ----------------
if speed_lora.startswith("Lightning 4"):
    light, auto_steps, auto_cfg = cfgfile.get("lightning4"), 4, 1.0
elif speed_lora.startswith("Lightning 8"):
    light, auto_steps, auto_cfg = cfgfile.get("lightning8"), 8, 1.0
    if not light:
        raise RuntimeError("The 8-step LoRA was not downloaded. Pick the 4-step option.")
else:
    light, auto_steps, auto_cfg = None, 40, 4.0
    print("⚠️ No speed LoRA: 40 steps with cfg 4 will take a very long time on a T4.")
eff_steps = int(steps) if int(steps) > 0 else auto_steps
eff_cfg = float(cfg) if float(cfg) > 0 else auto_cfg

# ---------------- prompt ----------------
full_prompt = prompt.strip()
if add_nsfw_trigger and use_nsfw_lora and "nsfw" not in full_prompt.lower():
    full_prompt = "nsfw, " + full_prompt
extra = STYLES[style]
if extra:
    full_prompt += ", " + extra

# ---------------- size ----------------
src_img, image_value = None, None
if edit_mode:
    image_value, src_img = upload_image(path)
    if aspect_ratio == "Match input image":
        W, H = size_from_image(src_img.width, src_img.height, megapixels)
    else:
        W, H = size_from_ratio(aspect_ratio, megapixels)
    mode_name = "IMAGE EDIT"
else:
    ratio = "1:1" if aspect_ratio == "Match input image" else aspect_ratio
    if aspect_ratio == "Match input image":
        print("ℹ️ No input image, so using 1:1.")
    W, H = size_from_ratio(ratio, megapixels)
    mode_name = "TEXT-TO-IMAGE"

print("Mode   :", mode_name)
print("Prompt :", full_prompt)
print("Size   : %dx%d | steps %d | cfg %s | NSFW LoRA %s | speed LoRA %s"
      % (W, H, eff_steps, eff_cfg, ("%.2f" % nsfw_lora_strength) if use_nsfw_lora else "off", light or "off"))
print("(First image loads the model into memory - can take several minutes. Later ones are faster.)\n")

# ---------------- helpers for the graph ----------------
def node_info(name):
    return requests.get(URL + "/object_info/" + name).json().get(name)

def combo_options(name, field):
    info = node_info(name)
    if not info:
        return []
    spec = info["input"].get("required", {}).get(field)
    if not spec:
        return []
    if isinstance(spec[0], list):
        return spec[0]
    if len(spec) > 1 and isinstance(spec[1], dict):
        return spec[1].get("options", [])
    return []

graph = {
    "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": cfgfile["unet"]}},
    "2": {"class_type": "CLIPLoaderGGUF", "inputs": {"clip_name": cfgfile["clip"], "type": "qwen_image"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": cfgfile["vae"]}},
}

def add(class_type, inputs):
    nid = str(len(graph) + 100)
    graph[nid] = {"class_type": class_type, "inputs": inputs}
    return nid

# --- model chain: base -> speed LoRA -> NSFW LoRA -> AuraFlow shift
model_ref = ["1", 0]
if light:
    n = add("LoraLoaderModelOnly", {"model": model_ref, "lora_name": light, "strength_model": 1.0})
    model_ref = [n, 0]
if use_nsfw_lora:
    n = add("LoraLoaderModelOnly", {"model": model_ref, "lora_name": cfgfile["nsfw_lora"],
                                    "strength_model": float(nsfw_lora_strength)})
    model_ref = [n, 0]
n = add("ModelSamplingAuraFlow", {"model": model_ref, "shift": float(aura_shift)})
model_ref = [n, 0]

# --- conditioning
image_ref = None
if edit_mode:
    graph["10"] = {"class_type": "LoadImage", "inputs": {"image": image_value}}
    image_ref = ["10", 0]

plus = node_info("TextEncodeQwenImageEditPlus")
plus_fields = {}
if plus:
    plus_fields = {**plus["input"].get("required", {}), **plus["input"].get("optional", {})}

def encode(text):
    """Text (+ reference image) -> conditioning, using whichever Qwen edit node this ComfyUI has."""
    if edit_mode and plus and "image1" in plus_fields:
        inp = {"clip": ["2", 0], "prompt": text, "image1": image_ref}
        if "vae" in plus_fields:
            inp["vae"] = ["3", 0]
        return add("TextEncodeQwenImageEditPlus", inp)
    inp = {"clip": ["2", 0], "prompt": text, "vae": ["3", 0]}
    if edit_mode:
        inp["image"] = image_ref
    return add("TextEncodeQwenImageEdit", inp)

use_ref_method = (edit_mode and node_info("FluxKontextMultiReferenceLatentMethod")
                  and "index_timestep_zero" in combo_options("FluxKontextMultiReferenceLatentMethod",
                                                             "reference_latents_method"))

def finish_cond(nid):
    ref = [nid, 0]
    if use_ref_method:
        m = add("FluxKontextMultiReferenceLatentMethod",
                {"conditioning": ref, "reference_latents_method": "index_timestep_zero"})
        ref = [m, 0]
    return ref

pos_ref = finish_cond(encode(full_prompt))
if eff_cfg > 1.0:
    neg_ref = finish_cond(encode(negative_prompt if negative_prompt else " "))
else:
    neg_ref = pos_ref   # cfg 1 ignores the negative anyway

graph["6"] = {"class_type": "EmptyLatentImage", "inputs": {"width": W, "height": H, "batch_size": 1}}
graph["7"] = {"class_type": "KSampler", "inputs": {
    "model": model_ref, "positive": pos_ref, "negative": neg_ref, "latent_image": ["6", 0],
    "seed": 0, "steps": eff_steps, "cfg": eff_cfg,
    "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0}}
graph["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
graph["9"] = {"class_type": "SaveImage", "inputs": {"images": ["8", 0],
              "filename_prefix": "qwen_lora_edit" if edit_mode else "qwen_lora_t2i"}}

# ---------------- run ----------------
try:
    log_start = os.path.getsize("/content/comfyui.log")
except Exception:
    log_start = 0

saved = []
for i in range(int(num_images)):
    s = (int(seed) + i) if int(seed) >= 0 else random.randint(0, 2**32 - 1)
    graph["7"]["inputs"]["seed"] = s
    t0 = time.time()
    r = requests.post(URL + "/prompt", json={"prompt": graph})
    if r.status_code != 200:
        print("❌ ComfyUI rejected the workflow:\n", r.text)
        raise RuntimeError("Workflow rejected — copy the message above and send it to get it fixed.")
    res = wait_for(r.json()["prompt_id"])
    for node_out in res["outputs"].values():
        for img in node_out.get("images", []):
            if img.get("type") == "temp":
                continue
            p = os.path.join(OUT_DIR, img.get("subfolder", ""), img["filename"])
            saved.append(p)
            print("✅ [%d/%d] seed %d | %.0fs | saved: %s" % (i + 1, int(num_images), s, time.time() - t0, p))
            out_im = Image.open(p).convert("RGB")
            if src_img is not None:
                print("   left = original, right = edited")
                display(side_by_side(src_img, out_im))
            else:
                pv = out_im.copy()
                pv.thumbnail((768, 768))
                display(pv)

# ---------------- did the LoRAs really apply? ----------------
try:
    with open("/content/comfyui.log", "r", errors="ignore") as f:
        f.seek(log_start)
        new_log = f.read()
    missed = new_log.count("lora key not loaded")
    if missed:
        print("\n⚠️ %d LoRA keys were NOT applied (see /content/comfyui.log). If the NSFW effect is missing or weak, "
              "the LoRA may not be compatible with this loader - copy the message and send it to get it fixed." % missed)
except Exception:
    pass

print("\nAll images are in", OUT_DIR, "(left sidebar -> 📁 Files -> outputs)")
